# 13.6 beta加大，單次推動一定更大嗎？


固定策略較佳−2、較差−4，參考兩篇都是−3，相對差距為2。此時已經偏向人標的較佳答案。只改beta，我們要看同一位置附近的代價與梯度，而不是比較重新訓練後的模型。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch
from tiny_perceptron.alignment import dpo_loss

for beta in [0.1, 1.0, 5.0]:
    chosen = torch.tensor([-2.0], requires_grad=True)
    loss = dpo_loss(chosen, torch.tensor([-4.0]), torch.tensor([-3.0]), torch.tensor([-3.0]), beta=beta)
    loss.backward()
    print(beta, "代價", round(loss.item(), 6), "梯度", round(chosen.grad.item(), 6))

0.1 代價 0.598139 梯度 -0.045017
1.0 代價 0.126928 梯度 -0.119203
5.0 代價 4.5e-05 梯度 -0.000227


每輪新建chosen，避免累積前輪梯度。三種beta的結果為：

| beta | 代價 | 較佳分數梯度 |
| ---: | ---: | ---: |
| 0.1 | 0.598139 | −0.045017 |
| 1 | 0.126928 | −0.119203 |
| 5 | 0.000045 | −0.000227 |

beta從0.1到1，梯度絕對值變大；到5反而變小。因為`sigmoid(beta×2)`已接近1，再增加差距的影響很小，這叫**飽和**。學習率固定時，這個局部更新幅度也按梯度絕對值變。

beta與學習率作用不同：beta先進入代價公式，會改變飽和程度；學習率最後才乘梯度。DPO的推導也把beta對應到偏離參考的代價係數，在固定獎勵尺度的理論目標中，較大beta對偏離懲罰較重。這不等於本表能預測完整訓練的最終行為。選設定仍要看驗證任務與原能力，beta必須大於0。

練習只把策略較佳改−6，使margin變−2。再觀察：當模型仍偏向較差答案，大beta的梯度不再因「已經做得夠好」而接近0。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

我們也從同一加法起點真正更新兩支模型，beta分別為0.1與1，每批八對、學習率固定0.001、各250步。寬度64、兩層、141,568個參數與資料、抽樣種子都相同。表中的「較佳排前」只比較完整chosen與rejected的機率；「相對改善」則比較策略與固定參考的差距，兩者不能互換。

| 設定 | 八題驗證：較佳排前 | 七題最後檢查：較佳排前 | 七題最後檢查：相對改善 | 七題實際生成答對 |
| --- | ---: | ---: | ---: | ---: |
| 更新前起點 | 0／8 | 1／7 | 0／7 | 0／7 |
| beta 0.1 | 3／8 | 1／7 | 1／7 | 0／7 |
| beta 1 | 0／8 | 0／7 | 0／7 | 0／7 |

驗證題中兩支各有5／8的相對差距改善，但最後檢查的平均相對差距分別為−16.27347與−5.03428，並沒有重現驗證側的進步。兩支在八題驗證及七題最後檢查都沒生成正確答案，全部都有EOS；例如`2+2=?`都回答`8`。這次beta 0.1在驗證候選排序上較好，不能據此稱它普遍較好，更不能把這一個種子、四個留出家族的失敗解讀為DPO必然無效。它讓我們看見這份小資料和弱起點尚未達成什麼。

</details>
